# Agente de IA para cotizar notebooks en SoloTodo usando LangChain

Requerimientos:

- Crear un archivo .env con los campos `OPENAI_API_KEY`

In [47]:
%pip install langchain langchain-openai requests python-dotenv > /dev/null

Note: you may need to restart the kernel to use updated packages.


# Cargar las credenciales de OpenAI

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
assert 'OPENAI_API_KEY' in os.environ

# Jupyter utilities

In [3]:
# Utility function to display messages
def pretty_print_message(message, max_chars: int = 2000) -> None:
    content = message.content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(block["text"])
        content = "\n\n".join(parts)

    if len(content) > max_chars:
        content = content[:max_chars - 1] + "…"

    message.model_copy(update={"content": content}).pretty_print()

# Parte 1: LLM básica

In [4]:
from langchain.chat_models import init_chat_model

In [5]:
llm = init_chat_model(model="gpt-6-luna")

In [6]:
# Para conocimiento general anda bien
pretty_print_message(llm.invoke('Qué es un notebook gamer?'))

================================== Ai Message ==================================

Un **notebook gamer** es una computadora portátil diseñada para jugar videojuegos. Suele tener una **tarjeta gráfica dedicada**, un procesador potente, más memoria y una pantalla con alta frecuencia de actualización para ofrecer gráficos fluidos.

También sirve para tareas exigentes, como edición de video o diseño 3D. A cambio, suele ser más cara, pesada y consumir más batería que una notebook común.


In [7]:
# Solo cuenta con los datos aproximados al momento de entrenarla
pretty_print_message(llm.invoke('Cuáles son las especificaciones del Lenovo IdeaPad Vibe 15AKP12 [83XNCTO1WWCL5]?'))

================================== Ai Message ==================================

No puedo confirmar una ficha técnica fiable para el **Lenovo IdeaPad Vibe 15AKP12, MTM 83XNCTO1WWCL5** solo con ese código. **CTO** suele indicar una configuración *Configure To Order*, por lo que el procesador, la memoria, el almacenamiento, la pantalla y el sistema operativo pueden variar.

Para obtener la configuración exacta, busca el MTM completo en **Lenovo Support** o en **Lenovo Parts Lookup**. Si me compartes el enlace de la publicación o una captura de la ficha, puedo resumirte las especificaciones y ayudarte a evaluarlo. No publiques tu número de serie.


In [51]:
# Las respuestas son genéricas
pretty_print_message(llm.invoke('Cual es el mejor notebook gamer por 1.5 MM listado en SoloTodo?'))

================================== Ai Message ==================================

Si te refieres a **$1.500.000 CLP**, no puedo ver en tiempo real qué ofertas aparecen hoy en SoloTodo, así que no quiero inventarte un modelo o precio exacto.

Como regla, elegiría **el notebook con RTX 4070 que no supere ese presupuesto**; si no hay uno conveniente, buscaría una **RTX 4060 de 8 GB**, idealmente en un **Lenovo LOQ** o **ASUS TUF**. Compara también el límite de potencia (TGP) de la GPU, la pantalla y la RAM: dos equipos con la misma RTX pueden rendir distinto.

Si me mandas los enlaces de SoloTodo —o una captura de los candidatos— te digo cuál conviene más.


In [9]:
# No tiene memoria...
pretty_print_message(llm.invoke('Mi nombre es Vijay'))

================================== Ai Message ==================================

¡Mucho gusto, Vijay! ¿En qué puedo ayudarte?


In [10]:
pretty_print_message(llm.invoke('Me podrías decir mi nombre?'))

================================== Ai Message ==================================

No me lo has dicho todavía. ¿Cómo te llamas?


# Parte 2: Nuestro primer agente

In [11]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

In [53]:
agent = create_agent(model=llm)

In [13]:
response = agent.invoke({"messages": [HumanMessage("Hola Agente, mi nombre es Vijay, mucho gusto, me podrías decir el nombre de un elemento químico interesante?")]})

In [14]:
messages = response['messages']
for message in messages:
    pretty_print_message(message)

================================ Human Message =================================

Hola Agente, mi nombre es Vijay, mucho gusto, me podrías decir el nombre de un elemento químico interesante?
================================== Ai Message ==================================

¡Mucho gusto, Vijay! Un elemento interesante es el **galio**: se derrite a unos 30 °C, así que puede fundirse con el calor de la mano.


In [15]:
# Puede guardar "memoria" en la medida que le enviemos toda la cadena de mensajes previa
messages.append(HumanMessage("Excelente, me podrías repetir mi nombre y el elemento químico que me dijiste?"))
response = agent.invoke({"messages": messages})
for message in response['messages']:
    pretty_print_message(message)

================================ Human Message =================================

Hola Agente, mi nombre es Vijay, mucho gusto, me podrías decir el nombre de un elemento químico interesante?
================================== Ai Message ==================================

¡Mucho gusto, Vijay! Un elemento interesante es el **galio**: se derrite a unos 30 °C, así que puede fundirse con el calor de la mano.
================================ Human Message =================================

Excelente, me podrías repetir mi nombre y el elemento químico que me dijiste?
================================== Ai Message ==================================

Tu nombre es Vijay y el elemento químico que te mencioné es el galio.


In [16]:
# Si no... se le olvidan las cosas
response = agent.invoke({"messages": [HumanMessage("Me podrías repetir nuevamente mi nombre?")]})
for message in response['messages']:
    pretty_print_message(message)

================================ Human Message =================================

Me podrías repetir nuevamente mi nombre?
================================== Ai Message ==================================

No me has dicho tu nombre en esta conversación. ¿Me lo recuerdas?


In [55]:
# Y aun no tiene información de datos actuales
response = agent.invoke({"messages": [HumanMessage('Cual es el mejor notebook gamer por 1.5 MM listado en SoloTodo?')]})
for message in response['messages']:
    pretty_print_message(message)

================================ Human Message =================================

Cual es el mejor notebook gamer por 1.5 MM listado en SoloTodo?
================================== Ai Message ==================================

No puedo consultar los precios de SoloTodo en tiempo real, así que no quiero inventarte un modelo que quizá ya no esté listado.

Como regla, por **$1.500.000 CLP**, elegiría el notebook con **RTX 5060** que encuentres dentro del presupuesto; si no hay ninguno, buscaría uno con **RTX 4060 de 8 GB**, al menos **16 GB de RAM** y pantalla de **144 Hz o más**. Un Lenovo LOQ o ASUS TUF con esa configuración suele ser una buena apuesta. Evitaría pagar ese monto por una RTX 4050.

Si me compartes el enlace de SoloTodo o una captura de los candidatos, te digo cuál conviene más entre esos modelos.


In [56]:
# No solicita información adicional al usuario
response = agent.invoke({"messages": [HumanMessage("Recomiendame un notebook")]})
for message in response['messages']:
    pretty_print_message(message)

================================ Human Message =================================

Recomiendame un notebook
================================== Ai Message ==================================

¡Claro! Para recomendarte un modelo concreto, dime **tu presupuesto**, **en qué país estás** y **para qué lo usarás** (estudio, trabajo, juegos, diseño, etc.).

Como recomendación general para estudiar y trabajar: busca un notebook de **14″**, con **16 GB de RAM**, **SSD de 512 GB** y un procesador **AMD Ryzen 5 o Intel Core 5** reciente. Evitaría los modelos con solo 8 GB de RAM si quieres que te dure varios años.


# Parte 3: Dándole herramientas al agente

In [19]:
import requests
import json

In [20]:
def obtain_processor_list():
    response = requests.get('https://api.solotodo.com/notebook_processors/')
    processors = [{'id': x['id'], 'name': x['unicode'], 'performance_score': x['speed_score']} for x in response.json()]
    return processors

In [21]:
print(json.dumps(obtain_processor_list()[:5], indent=2))

[
  {
    "id": 115856,
    "name": "AMD A6-6310",
    "performance_score": 2483
  },
  {
    "id": 115889,
    "name": "AMD A6-7050B",
    "performance_score": 1727
  },
  {
    "id": 646153,
    "name": "AMD A6-9220",
    "performance_score": 1935
  },
  {
    "id": 666036,
    "name": "AMD A10-8730B",
    "performance_score": 4075
  },
  {
    "id": 508664,
    "name": "AMD A10-9600P",
    "performance_score": 3807
  }
]


In [22]:
from langchain.tools import tool

In [23]:
@tool
def processor_list_tool() -> list[dict]:
    """Returns the list of notebook processors listed in SoloTodo"""
    return obtain_processor_list()

In [24]:
agent = create_agent(
    model=llm,
    tools=[processor_list_tool]
)

In [25]:
response = agent.invoke({"messages": [HumanMessage("Cuál es el procesador de notebook más rápido listado en SoloTodo?")]})
for message in response['messages']:
    pretty_print_message(message)

================================ Human Message =================================

Cuál es el procesador de notebook más rápido listado en SoloTodo?
================================== Ai Message ==================================
Tool Calls:
  processor_list_tool (call_dQmGTzBRImRWiTdBjoSSpcbv)
 Call ID: call_dQmGTzBRImRWiTdBjoSSpcbv
  Args:
================================= Tool Message =================================
Name: processor_list_tool

[{"id": 115856, "name": "AMD A6-6310", "performance_score": 2483}, {"id": 115889, "name": "AMD A6-7050B", "performance_score": 1727}, {"id": 646153, "name": "AMD A6-9220", "performance_score": 1935}, {"id": 666036, "name": "AMD A10-8730B", "performance_score": 4075}, {"id": 508664, "name": "AMD A10-9600P", "performance_score": 3807}, {"id": 648133, "name": "AMD A10-9620P", "performance_score": 4480}, {"id": 1139737, "name": "AMD Athlon Silver 3050U", "performance_score": 3151}, {"id": 116342, "name": "AMD Athlon 64 X2 TK-55", "performance_scor

In [75]:
@tool
def video_card_list_tool() -> list[dict]:
    """Returns the list of notebook video cards listed in SoloTodo"""
    response = requests.get('https://api.solotodo.com/notebook_video_cards/')
    video_cards = [{'id': x['id'], 'name': x['unicode'], 'performance_score': x['speed_score']} for x in response.json()]
    return video_cards

In [27]:
agent = create_agent(
    model=llm,
    tools=[processor_list_tool, video_card_list_tool]
)

In [28]:
response = agent.invoke({"messages": [HumanMessage("Cuál es la tarjeta de video para notebooks fabricada por AMD más rápida en SoloTodo?")]})
for message in response['messages']:
    pretty_print_message(message)

================================ Human Message =================================

Cuál es la tarjeta de video para notebooks fabricada por AMD más rápida en SoloTodo?
================================== Ai Message ==================================
Tool Calls:
  video_card_list_tool (call_SqL9c3k92jHYrrugg3yFAm7U)
 Call ID: call_SqL9c3k92jHYrrugg3yFAm7U
  Args:
================================= Tool Message =================================
Name: video_card_list_tool

[{"id": 108117, "name": "AMD Radeon R5 M335 (2 GB)", "performance_score": 911}, {"id": 525391, "name": "AMD Radeon R6 (Carrizo) (Integrada)", "performance_score": 1022}, {"id": 108147, "name": "AMD Radeon R7 M265 (2 GB)", "performance_score": 1336}, {"id": 553376, "name": "AMD Radeon R7 M440 (2 GB)", "performance_score": 1205}, {"id": 531686, "name": "AMD Radeon R8 Graphics (Carrizo) (Integrada)", "performance_score": 0}, {"id": 517314, "name": "AMD Radeon R9 M365X (1 GB)", "performance_score": 0}, {"id": 645717, "name": "

In [29]:
@tool
def browse_notebooks(processor_ids: list[int] | None, video_card_ids: list[int] | None, min_ram_quantity: int | None, max_usd_price: float | None, page: int) -> list:
    """Returns the notebooks from SoloTodo filtered by the given params and sorted by price
        Args:

        processor_ids: IDs of the processors as returned from the processor_list_tool tool
        video_card_ids: IDs of the video cards as returned from the video_card_list_tool tool
        min_ram_quantity: Minimum RAM quantity (in GB)
        max_usd_price: Maximum price (in USD). For conversion you may assume an exchange rate of 994.58 between USD and CLP
        page: Results page number (starts at 1)
    """
    endpoint = f'https://api.solotodo.com/categories/1/browse/?page_size=200&ordering=offer_price_usd&exclude_refurbished=1&page={page}'

    if processor_ids:
        for processor_id in processor_ids:
            endpoint += f"&processors={processor_id}"
            
    if video_card_ids:
        for video_card_id in video_card_ids:
            endpoint += f"&video_cards={video_card_id}"

    if min_ram_quantity:
        endpoint += f"&ram_quantity_value_min={min_ram_quantity}"

    if max_usd_price:
        endpoint += f"&offer_price_usd_max={max_usd_price}"

    response = requests.get(endpoint)
    return response.json()

In [30]:
system_prompt = """
    Eres un asistente de compras para notebooks dentro del sitio de SoloTodo. 
    Tu trabajo es responder con recomendaciones de equipos según lo que te indique el usuario.
    Si te indican precios, asume que están en pesos chilenos (CLP), para convertirlos a dólares puedes usar la tasa de conversión 994.58 USD-CLP.
    Sólo responde dudas relacionadas con cotización y compra de notebooks
    Utiliza solamente los precios y productos listados en SoloTodo
    Si no puedes tomar una decisión, o si no hay productos que cumplan con las condiciones que te solicitan, responde con un mensaje acorde
    No inventes especificaciones o precios
    """

In [31]:
agent = create_agent(
    model=llm,
    tools=[processor_list_tool, video_card_list_tool, browse_notebooks],
    system_prompt=system_prompt
)

In [32]:
response = agent.invoke({"messages": [HumanMessage("Recomiéndame notebooks para juegos con un precio de hasta 1.600.000")]})
for message in response['messages']:
    pretty_print_message(message)

================================ Human Message =================================

Recomiéndame notebooks para juegos con un precio de hasta 1.600.000
================================== Ai Message ==================================
Tool Calls:
  processor_list_tool (call_uV6eyfhWF7kVAeIgIdlsYfPa)
 Call ID: call_uV6eyfhWF7kVAeIgIdlsYfPa
  Args:
  video_card_list_tool (call_CQe5V1RC7kaY5x86GcIARUFR)
 Call ID: call_CQe5V1RC7kaY5x86GcIARUFR
  Args:
================================= Tool Message =================================
Name: processor_list_tool

[{"id": 115856, "name": "AMD A6-6310", "performance_score": 2483}, {"id": 115889, "name": "AMD A6-7050B", "performance_score": 1727}, {"id": 646153, "name": "AMD A6-9220", "performance_score": 1935}, {"id": 666036, "name": "AMD A10-8730B", "performance_score": 4075}, {"id": 508664, "name": "AMD A10-9600P", "performance_score": 3807}, {"id": 648133, "name": "AMD A10-9620P", "performance_score": 4480}, {"id": 1139737, "name": "AMD Athlon Silv

In [33]:
response = agent.invoke({"messages": [HumanMessage("Notebook para estudiar arquitectura")]})
for message in response['messages']:
    pretty_print_message(message)

================================ Human Message =================================

Notebook para estudiar arquitectura
================================== Ai Message ==================================
Tool Calls:
  processor_list_tool (call_EAxpgRf3SyrUEHZIykOzKm5H)
 Call ID: call_EAxpgRf3SyrUEHZIykOzKm5H
  Args:
  video_card_list_tool (call_vEcA6nQpgMozDoz8pxLI5PUW)
 Call ID: call_vEcA6nQpgMozDoz8pxLI5PUW
  Args:
================================= Tool Message =================================
Name: processor_list_tool

[{"id": 115856, "name": "AMD A6-6310", "performance_score": 2483}, {"id": 115889, "name": "AMD A6-7050B", "performance_score": 1727}, {"id": 646153, "name": "AMD A6-9220", "performance_score": 1935}, {"id": 666036, "name": "AMD A10-8730B", "performance_score": 4075}, {"id": 508664, "name": "AMD A10-9600P", "performance_score": 3807}, {"id": 648133, "name": "AMD A10-9620P", "performance_score": 4480}, {"id": 1139737, "name": "AMD Athlon Silver 3050U", "performance_score": 

# Parte 4: Solicitar información adicional al usuario

In [76]:
from uuid import uuid4  # Para darle un ID único a la invocación
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver

In [77]:
@tool
def ask_user(question: str) -> str:
    """Pide al comprador una aclaración necesaria para recomendar notebooks."""
    return interrupt({"question": question})

In [78]:
agent = create_agent(
    model=llm,
    tools=[processor_list_tool, video_card_list_tool, browse_notebooks, ask_user],
    system_prompt=system_prompt + """
    Si falta un dato que cambiaría sustancialmente la búsqueda, usa ask_user.
    Por ejemplo, si el usuario busca un notebook gamer sin indicar presupuesto,
    pregunta cuánto quiere gastar como máximo en CLP.
    Haz una sola pregunta concreta y espera la respuesta antes de llamar
    a browse_notebooks. Si ya tienes suficiente información, busca directamente.
    """,
    checkpointer=InMemorySaver(),
)

In [80]:
thread_id = str(uuid4())
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Notebook para estudiar arquitectura"}]},
    config={"configurable": {"thread_id": thread_id}}
)
for message in result['messages']:
    pretty_print_message(message)

================================ Human Message =================================

Notebook para estudiar arquitectura
================================== Ai Message ==================================
Tool Calls:
  ask_user (call_WN3KUWhJjLl4xTv8YjvreDrS)
 Call ID: call_WN3KUWhJjLl4xTv8YjvreDrS
  Args:
    question: ¿Cuál es tu presupuesto máximo en pesos chilenos (CLP) para el notebook?


In [81]:
def handle_agent_interruptions(result, thread_id):
    # "Mientras queden interrupciones pendientes"
    while result.get("__interrupt__"):
        # Obtener el motivo de la interrupción
        pending, = result["__interrupt__"]
        # Obtener la respuesta del usuario a la pregunta
        answer = input(pending.value["question"] + " ")
        # Actualizar el valor de la variable result con la ejecución del agente pasando la respuesta 
        result = agent.invoke(Command(resume=answer), config={"configurable": {"thread_id": thread_id}})
    return result

In [82]:
result = handle_agent_interruptions(result, thread_id)

¿Cuál es tu presupuesto máximo en pesos chilenos (CLP) para el notebook?  1000000


In [83]:
for message in result['messages']:
    pretty_print_message(message)

================================ Human Message =================================

Notebook para estudiar arquitectura
================================== Ai Message ==================================
Tool Calls:
  ask_user (call_WN3KUWhJjLl4xTv8YjvreDrS)
 Call ID: call_WN3KUWhJjLl4xTv8YjvreDrS
  Args:
    question: ¿Cuál es tu presupuesto máximo en pesos chilenos (CLP) para el notebook?
================================= Tool Message =================================
Name: ask_user

1000000
================================== Ai Message ==================================
Tool Calls:
  processor_list_tool (call_5bTusL4pAtH4SHAsrr1zoiPj)
 Call ID: call_5bTusL4pAtH4SHAsrr1zoiPj
  Args:
  video_card_list_tool (call_kfQXZkFBc5geaYuzPx2i6N2G)
 Call ID: call_kfQXZkFBc5geaYuzPx2i6N2G
  Args:
================================= Tool Message =================================
Name: processor_list_tool

[{"id": 115856, "name": "AMD A6-6310", "performance_score": 2483}, {"id": 115889, "name": "AMD A6-70

In [41]:
thread_id = str(uuid4())
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Recomiendame un notebook"}]},
    config={"configurable": {"thread_id": thread_id}}
)
result = handle_agent_interruptions(result, thread_id)
for message in result['messages']:
    pretty_print_message(message)

¿Para qué lo usarás principalmente (estudio/oficina, gaming, diseño u otro) y cuál es tu presupuesto máximo en CLP?  para oficina, cosas ligeras
¿Cuál es tu presupuesto máximo en CLP para el notebook de oficina?  500000


================================ Human Message =================================

Recomiendame un notebook
================================== Ai Message ==================================
Tool Calls:
  ask_user (call_okO5mg5VFEGzhxHp9TQaL0aZ)
 Call ID: call_okO5mg5VFEGzhxHp9TQaL0aZ
  Args:
    question: ¿Para qué lo usarás principalmente (estudio/oficina, gaming, diseño u otro) y cuál es tu presupuesto máximo en CLP?
================================= Tool Message =================================
Name: ask_user

para oficina, cosas ligeras
================================== Ai Message ==================================
Tool Calls:
  ask_user (call_YGcOuwBcVWs0Zu7iAg06mXqD)
 Call ID: call_YGcOuwBcVWs0Zu7iAg06mXqD
  Args:
    question: ¿Cuál es tu presupuesto máximo en CLP para el notebook de oficina?
================================= Tool Message =================================
Name: ask_user

500000
================================== Ai Message ==================================
Tool

# Parte 5: Integrar herramientas externas

Requerimientos:

- Agregar el campo `TAVILY_API_KEY` a nuestro .env

In [69]:
%pip install langchain-tavily > /dev/null

Note: you may need to restart the kernel to use updated packages.


In [70]:
assert 'TAVILY_API_KEY' in os.environ

In [71]:
from langchain_tavily import TavilySearch

In [72]:
tavily_search = TavilySearch()

In [73]:
agent = create_agent(
    model=llm,
    tools=[
        processor_list_tool,
        video_card_list_tool,
        browse_notebooks,
        ask_user,
        tavily_search,
    ],
    system_prompt=system_prompt + """
    Si falta un dato que cambiaría sustancialmente la búsqueda, usa ask_user.
    Por ejemplo, si el usuario busca un notebook gamer sin indicar presupuesto,
    pregunta cuánto quiere gastar como máximo en CLP.
    Haz una sola pregunta concreta y espera la respuesta antes de llamar
    a browse_notebooks. Si ya tienes suficiente información, busca directamente.

    Usa Tavily para consultar benchmarks o comparativas de componentes cuando
    el rendimiento sea importante para decidir qué notebooks recomendar.
    Para GPUs, busca específicamente benchmarks de sus versiones Laptop.
    Si las fuentes indican la potencia (TGP), considérala al comparar resultados.

    Usa los resultados de Tavily para decidir qué componentes priorizar.
    Usa los nombres de filtros del catálogo al llamar a browse_notebooks.
    Obtén los modelos que recomiendas, sus precios y su disponibilidad
    exclusivamente de browse_notebooks.
    Si el usuario ya indicó filtros concretos y solo quiere cotizar, llama
    directamente a browse_notebooks.
    """,
    checkpointer=InMemorySaver(),
)

In [74]:
thread_id = str(uuid4())
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Recomiendame un notebook con un rendimiento similar an equipo desktop equipado con una RTX 5060"}]},
    config={"configurable": {"thread_id": thread_id}}
)
result = handle_agent_interruptions(result, thread_id)
for message in result['messages']:
    pretty_print_message(message)

¿Cuál es tu presupuesto máximo en pesos chilenos (CLP) para el notebook?  2000000


================================ Human Message =================================

Recomiendame un notebook con un rendimiento similar an equipo desktop equipado con una RTX 5060
================================== Ai Message ==================================
Tool Calls:
  ask_user (call_EDqZJD2IdrNmp9s1Ugc4plgn)
 Call ID: call_EDqZJD2IdrNmp9s1Ugc4plgn
  Args:
    question: ¿Cuál es tu presupuesto máximo en pesos chilenos (CLP) para el notebook?
================================= Tool Message =================================
Name: ask_user

2000000
================================== Ai Message ==================================
Tool Calls:
  processor_list_tool (call_buEdPMjOJNYXFKnKejSEnM6m)
 Call ID: call_buEdPMjOJNYXFKnKejSEnM6m
  Args:
  video_card_list_tool (call_hrOPbtH82Vj8UgZDxltQM1ww)
 Call ID: call_hrOPbtH82Vj8UgZDxltQM1ww
  Args:
  tavily_search (call_i6k6SpT5x3c23vFInm9gxScA)
 Call ID: call_i6k6SpT5x3c23vFInm9gxScA
  Args:
    query: RTX 5060 desktop vs laptop GPU gaming ben